In [1]:
import pandas as pd

# Kiểm tra file Việt
df_vi = pd.read_csv('train_vi.csv')
print("Cột Tiếng Việt:", df_vi.columns)

# Kiểm tra file Anh
df_en = pd.read_csv('train_en.csv')
print("Cột Tiếng Anh:", df_en.columns)

Cột Tiếng Việt: Index(['free_text', 'label_id'], dtype='object')
Cột Tiếng Anh: Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')


## Xử lý file tiếng Việt và file tiếng anh

In [4]:
import pandas as pd

# Đọc file gốc
df_en = pd.read_csv('train_en.csv')

# 1. Chỉ giữ lại 2 cột cần thiết
df_en = df_en[['comment_text', 'toxic']]

# 2. Đổi tên cột cho dễ quản lý (tùy chọn)
df_en.columns = ['text', 'label']

# 3. Lấy mẫu (Sampling) - Khoảng 150,000 dòng để cân bằng với bộ Việt
# Việc này giúp file model .pkl sau này nhẹ hơn, Azure 4GB load nhanh hơn
df_en = df_en.sample(n=150000, random_state=42)

# 4. Ghi đè lại file
df_en.to_csv('en_train_cleaned.csv', index=False, encoding='utf-8')
print("✅ Đã xử lý xong file Anh: Còn 2 cột, 150k dòng.")

✅ Đã xử lý xong file Anh: Còn 2 cột, 150k dòng.


In [5]:
df_vi = pd.read_csv('train_vi.csv')

# 1. Chỉ giữ lại cột văn bản và nhãn
df_vi = df_vi[['free_text', 'label_id']]
df_vi.columns = ['text', 'label']

# 2. Mapping nhãn: 1 (Offensive) và 2 (Hate) -> đều là 1 (Toxic)
df_vi['label'] = df_vi['label'].map({0: 0, 1: 1, 2: 1})

# 3. Ghi đè lại file
df_vi.to_csv('vi_train_cleaned.csv', index=False, encoding='utf-8-sig')
print("✅ Đã xử lý xong file Việt: Nhãn đã đưa về 0 và 1.")

✅ Đã xử lý xong file Việt: Nhãn đã đưa về 0 và 1.


## Tiền xử lý dữ liệu

In [26]:
# ==========================================
# CELL 1: TIỀN XỬ LÝ NLP & LOAD DICTIONARY
# ==========================================
import pandas as pd
import re
import json
import unicodedata
from pyvi import ViTokenizer

# 1. Khai báo đường dẫn và hàm đọc file JSON
BADWORDS_PATH = "/media/haduckien/E/Studying/HK6/python_programming/project/apps/moderation/datasets/vi_badwords.json"

def load_toxic_json(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        toxic_set = set()
        if isinstance(data, list):
            toxic_set = set([str(w).lower().strip().replace(' ', '_') for w in data])
        elif isinstance(data, dict):
            toxic_set = set([str(k).lower().strip().replace(' ', '_') for k in data.keys()])
            
        print(f"✅ Đã load thành công {len(toxic_set)} từ khóa từ file JSON.")
        return toxic_set
    except Exception as e:
        print(f"⚠️ Lỗi đọc file JSON: {e}. Sẽ tiếp tục với tập rỗng.")
        return set()

TOXIC_VI = load_toxic_json(BADWORDS_PATH)

# 2. Hàm Tiền xử lý văn bản chuyên sâu
def advanced_nlp_preprocess(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    
    # 2.1 Chuẩn hóa Unicode (NFC)
    text = unicodedata.normalize('NFC', text)
    
    # 2.2 Loại bỏ URL, Email, Mentions
    text = re.sub(r'http\S+|@\S+|#\S+', ' ', text)
    
    # 2.3 Gom gọn các ký tự kéo dài (ví dụ: đmmmmm -> đmm)
    text = re.sub(r'([a-z])\1+', r'\1\1', text)
    
    # 2.4 Xóa ký tự đặc biệt, nhưng giữ lại dấu chấm hỏi và chấm than (!?)
    text = re.sub(r'[^\w\s!?]', ' ', text)
    
    # 2.5 Tách từ Tiếng Việt (Word Segmentation)
    text = ViTokenizer.tokenize(text.strip())
    
    return text

# ---> CHẠY THỬ VỚI DỮ LIỆU CỦA BẠN:
df_train = pd.read_csv('vi_train_cleaned.csv')
df_train['clean_text'] = df_train['text'].apply(advanced_nlp_preprocess)
print("Đã tiền xử lý xong tập huấn luyện!")

✅ Đã load thành công 67 từ khóa từ file JSON.
Đã tiền xử lý xong tập huấn luyện!


In [27]:
# ==========================================
# CELL 2: KIẾN TRÚC FEATURE UNION & BASE PIPELINE
# ==========================================
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

print("⚙️ Đang khởi tạo kiến trúc trích xuất đặc trưng...")

# 1. Trích xuất CỤM TỪ (Word N-grams)
word_features = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),        # Bắt cụm 2 từ
    max_features=12000,        # Giới hạn chiều dữ liệu
    sublinear_tf=True,         # Giảm nhiễu khi lặp từ
    min_df=3,                  # Lọc từ xuất hiện quá ít
    max_df=0.85                # Tự động lọc từ vô nghĩa (Stopwords tự nhiên)
)

# 2. Trích xuất CỤM KÝ TỰ (Char N-grams)
char_features = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),        # Quét cụm 2-4 ký tự để diệt teencode
    max_features=6000,
    sublinear_tf=True
)

# 3. Gộp đặc trưng (Tổng cộng 18,000 chiều)
advanced_features = FeatureUnion([
    ('word_tfidf', word_features),
    ('char_tfidf', char_features)
])

# 4. Khởi tạo Base Pipeline (Chưa dùng Calibrated ở đây để GridSearch chạy nhanh)
base_pipeline = Pipeline([
    ('features', advanced_features),
    ('clf', LinearSVC(max_iter=3000))
])

print("✅ Đã thiết lập xong Base Pipeline (Word + Char TF-IDF -> LinearSVC).")

⚙️ Đang khởi tạo kiến trúc trích xuất đặc trưng...
✅ Đã thiết lập xong Base Pipeline (Word + Char TF-IDF -> LinearSVC).


In [28]:
# ==========================================
# CELL 3: GRIDSEARCHCV, HIỆU CHUẨN VÀ ĐÁNH GIÁ
# ==========================================
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, f1_score
import joblib

# 1. Chia tập Train/Test (Đảm bảo tỷ lệ nhãn phân bố đều bằng stratify)
X_train, X_test, y_train, y_test = train_test_split(
    df_train['clean_text'], 
    df_train['label'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df_train['label'] 
)

# 2. Khai báo Lưới siêu tham số (Hyperparameter Grid)
param_grid = {
    'clf__C': [0.1, 0.5, 1.0, 5.0],
    'clf__class_weight': [
        'balanced', 
        {0: 1, 1: 1.5}, 
        {0: 1, 1: 2.0}
    ]
}

print("🕵️ Đang chạy GridSearchCV để tìm tham số tối ưu (Vui lòng đợi)...")
# 3. Khởi chạy GridSearch
grid_search = GridSearchCV(
    estimator=base_pipeline, 
    param_grid=param_grid, 
    cv=3, 
    scoring='f1_macro', 
    n_jobs=-1, # Dùng full nhân CPU
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n🎯 BỘ THAM SỐ ĐỈNH NHẤT:")
print(grid_search.best_params_)

# 4. Hiệu chuẩn xác suất (Platt Scaling) cho mô hình tốt nhất
print("\n⚙️ Đang hiệu chuẩn xác suất (Calibrating)...")
best_model = grid_search.best_estimator_
final_model = CalibratedClassifierCV(best_model, cv=3)
final_model.fit(X_train, y_train)

# 5. Đánh giá trên tập Test
y_pred = final_model.predict(X_test)

print("\n" + "="*40)
print("🏆 KẾT QUẢ ĐÁNH GIÁ CHUNG CUỘC 🏆")
print("="*40)
print(classification_report(y_test, y_pred, target_names=['An toàn (0)', 'Toxic (1)']))

print(f"🎯 F1-Score (Macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"🔥 F1-Score (Riêng Toxic): {f1_score(y_test, y_pred, pos_label=1):.4f}")

# 6. Đóng gói mô hình
MODEL_PATH = "nlp_toxic_model_vi_final.pkl"
joblib.dump(final_model, MODEL_PATH, compress=3)
print(f"\n📦 Đã xuất xưởng mô hình thành công: {MODEL_PATH}")

🕵️ Đang chạy GridSearchCV để tìm tham số tối ưu (Vui lòng đợi)...
Fitting 3 folds for each of 12 candidates, totalling 36 fits



🎯 BỘ THAM SỐ ĐỈNH NHẤT:
{'clf__C': 0.1, 'clf__class_weight': {0: 1, 1: 2.0}}

⚙️ Đang hiệu chuẩn xác suất (Calibrating)...

🏆 KẾT QUẢ ĐÁNH GIÁ CHUNG CUỘC 🏆
              precision    recall  f1-score   support

 An toàn (0)       0.92      0.97      0.94      3978
   Toxic (1)       0.79      0.58      0.67       832

    accuracy                           0.90      4810
   macro avg       0.85      0.77      0.80      4810
weighted avg       0.89      0.90      0.89      4810

🎯 F1-Score (Macro): 0.8041
🔥 F1-Score (Riêng Toxic): 0.6667

📦 Đã xuất xưởng mô hình thành công: nlp_toxic_model_vi_final.pkl


## test model

In [35]:
import pandas as pd
import joblib
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import unicodedata
import re
from pyvi import ViTokenizer

# 1. Tái định nghĩa hàm tiền xử lý (BẮT BUỘC phải giống lúc train)
def advanced_nlp_preprocess(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'http\S+|@\S+|#\S+', ' ', text)
    text = re.sub(r'([a-z])\1+', r'\1\1', text)
    text = re.sub(r'[^\w\s!?]', ' ', text)
    text = ViTokenizer.tokenize(text.strip())
    return text

# 2. Load mô hình đã lưu
MODEL_PATH = "nlp_toxic_model_vi_final.pkl"
try:
    final_model = joblib.load(MODEL_PATH)
    print(f"✅ Đã load mô hình thành công từ: {MODEL_PATH}")
except Exception as e:
    print(f"❌ Lỗi khi load model: {e}")

# 3. Đọc và xử lý tập test_vi.csv
TEST_FILE = "test_vi.csv" # Đảm bảo file này nằm cùng thư mục hoặc đúng đường dẫn
try:
    df_test = pd.read_csv(TEST_FILE)
    
    # Giả sử file test có cột 'text' và 'label' (hoặc bạn đổi tên cho đúng)
    # Nếu nhãn là [0, 1, 2] thì phải map về [0, 1] như lúc train
    if 'label_id' in df_test.columns:
        df_test['label'] = df_test['label_id'].map({0: 0, 1: 1, 2: 1})
    
    print(f"⏳ Đang tiền xử lý {len(df_test)} dòng dữ liệu test...")
    df_test['clean_text'] = df_test['free_text'].apply(advanced_nlp_preprocess)
    
    # 4. Thực hiện dự đoán
    print("🚀 Đang chạy dự đoán (Inference)...")
    y_true = df_test['label']
    y_pred = final_model.predict(df_test['clean_text'])
    
    # Lấy thêm xác suất (%) để kiểm tra độ tự tin của model
    y_proba = final_model.predict_proba(df_test['clean_text'])[:, 1]
    
    # 5. Xuất kết quả chi tiết
    print("\n" + "="*50)
    print(f"📊 KẾT QUẢ KIỂM THỬ TRÊN FILE: {TEST_FILE}")
    print("="*50)
    print(classification_report(y_true, y_pred, target_names=['An toàn (0)', 'Toxic (1)']))
    
    print(f"🎯 F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
    
    # Lưu kết quả dự đoán ra file để bạn soi lỗi (nếu cần)
    df_test['ai_prediction'] = y_pred
    df_test['ai_confidence'] = y_proba
    df_test.to_csv("test_vi_results.csv", index=False, encoding='utf-8-sig')
    print(f"\n📂 Đã lưu chi tiết kết quả dự đoán vào: test_vi_results.csv")

except FileNotFoundError:
    print(f"❌ Không tìm thấy file {TEST_FILE}. Hãy kiểm tra lại đường dẫn.")
except Exception as e:
    print(f"❌ Có lỗi xảy ra: {e}")

✅ Đã load mô hình thành công từ: nlp_toxic_model_vi_final.pkl
⏳ Đang tiền xử lý 6680 dòng dữ liệu test...
🚀 Đang chạy dự đoán (Inference)...

📊 KẾT QUẢ KIỂM THỬ TRÊN FILE: test_vi.csv
              precision    recall  f1-score   support

 An toàn (0)       0.91      0.97      0.94      5548
   Toxic (1)       0.78      0.53      0.63      1132

    accuracy                           0.90      6680
   macro avg       0.85      0.75      0.79      6680
weighted avg       0.89      0.90      0.89      6680

🎯 F1-Score (Macro): 0.7860

📂 Đã lưu chi tiết kết quả dự đoán vào: test_vi_results.csv


In [10]:
import joblib
import numpy as np

# ==========================================
# 1. LOAD MODEL
# ==========================================
MODEL_PATH = "nlp_toxic_model_vi_final.pkl"
model = joblib.load(MODEL_PATH)

print("✅ Load model thành công!")

# ==========================================
# 2. HÀM PREDICT 1 TEXT
# ==========================================
def predict_text(text, threshold=0.5):
    proba = model.predict_proba([text])[0]
    label = int(proba[1] >= threshold)

    return {
        "text": text,
        "label": label,              # 0 = an toàn, 1 = toxic
        "prob_safe": float(proba[0]),
        "prob_toxic": float(proba[1]),
        "threshold": threshold
    }

# ==========================================
# 3. TEST THỬ 1 CÂU
# ==========================================
test_text = "Na là một người học dốt"  # Câu an toàn

result = predict_text(test_text)

print("\n📌 INPUT:", result["text"])
print("🧠 LABEL:", result["label"])
print("🟢 Prob Safe:", round(result["prob_safe"], 4))
print("🔴 Prob Toxic:", round(result["prob_toxic"], 4))
print("⚙️ Threshold:", result["threshold"])

✅ Load model thành công!

📌 INPUT: Na là một người học dốt
🧠 LABEL: 0
🟢 Prob Safe: 0.9056
🔴 Prob Toxic: 0.0944
⚙️ Threshold: 0.5


## Tiếng anh

In [29]:
# ==========================================
# CELL 1: ENGLISH NLP PREPROCESSING
# ==========================================
import pandas as pd
import re
import unicodedata

def preprocess_english(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    
    # 1. Unicode Normalization
    text = unicodedata.normalize('NFC', text)
    
    # 2. Xóa URL, Email, Mentions
    text = re.sub(r'http\S+|@\S+|#\S+', ' ', text)
    
    # 3. Gom ký tự lặp (ví dụ: gooooood -> good, fffuck -> ffuck)
    text = re.sub(r'([a-z])\1+', r'\1\1', text)
    
    # 4. Giữ lại chữ, số và (!?) 
    text = re.sub(r'[^\w\s!?]', ' ', text)
    
    return text.strip()

# ---> CHẠY THỬ VỚI DỮ LIỆU ANH (Jigsaw 35k dòng):
df_en = pd.read_csv('en_train_cleaned.csv')
df_en['clean_text'] = df_en['text'].apply(preprocess_english)
print("✅ Đã tiền xử lý xong tập huấn luyện Tiếng Anh!")

✅ Đã tiền xử lý xong tập huấn luyện Tiếng Anh!


In [31]:
# ==========================================
# CELL 2: ENGLISH FEATURE UNION
# ==========================================
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# 1. Word Features (English)
word_features_en = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True,
    min_df=3,
    max_df=0.9
)

# 2. Char Features (English) - Rất mạnh để bắt Slang lách luật
char_features_en = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=5000,
    sublinear_tf=True
)

advanced_features_en = FeatureUnion([
    ('word_tfidf', word_features_en),
    ('char_tfidf', char_features_en)
])

# 3. Base Pipeline
base_pipeline_en = Pipeline([
    ('features', advanced_features_en),
    ('clf', LinearSVC(max_iter=3000))
])

print("✅ Đã thiết lập xong Pipeline Tiếng Anh.")

✅ Đã thiết lập xong Pipeline Tiếng Anh.


In [32]:
# ==========================================
# CELL 3: ENGLISH GRIDSEARCH & EXPORT
# ==========================================
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, f1_score
import joblib

# 1. Chia tập Train/Test
X_train_en, X_test_en, y_train_en, y_test_en = train_test_split(
    df_en['clean_text'], 
    df_en['label'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df_en['label'] 
)

# 2. Tham số quét (C cho tiếng Anh thường nhạy ở mức 0.1 - 1.0)
param_grid_en = {
    'clf__C': [0.1, 0.5, 1.0],
    'clf__class_weight': ['balanced', {0: 1, 1: 1.5}, {0: 1, 1: 2.0}]
}

print("🕵️ Đang tìm tham số tối ưu cho Tiếng Anh...")
grid_search_en = GridSearchCV(
    base_pipeline_en, param_grid_en, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1
)

grid_search_en.fit(X_train_en, y_train_en)

# 3. Hiệu chuẩn và Đánh giá
best_model_en = grid_search_en.best_estimator_
final_model_en = CalibratedClassifierCV(best_model_en, cv=3)
final_model_en.fit(X_train_en, y_train_en)

y_pred_en = final_model_en.predict(X_test_en)

print("\n🎯 BEST PARAMS (EN):", grid_search_en.best_params_)
print("\n" + "="*40)
print("🏆 KẾT QUẢ MÔ HÌNH TIẾNG ANH 🏆")
print("="*40)
print(classification_report(y_test_en, y_pred_en))

# 4. Lưu Model
joblib.dump(final_model_en, "nlp_toxic_model_en_final.pkl", compress=3)
print("\n📦 Đã xuất xưởng: nlp_toxic_model_en_final.pkl")

🕵️ Đang tìm tham số tối ưu cho Tiếng Anh...
Fitting 3 folds for each of 9 candidates, totalling 27 fits

🎯 BEST PARAMS (EN): {'clf__C': 0.1, 'clf__class_weight': {0: 1, 1: 2.0}}

🏆 KẾT QUẢ MÔ HÌNH TIẾNG ANH 🏆
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     27127
           1       0.89      0.69      0.78      2873

    accuracy                           0.96     30000
   macro avg       0.93      0.84      0.88     30000
weighted avg       0.96      0.96      0.96     30000


📦 Đã xuất xưởng: nlp_toxic_model_en_final.pkl
